# XAI-MedCrossNet++ | VinDr-Mammo | Production Notebook

**Real dataset pipeline — all root-causes fixed:**
- Real BI-RADS label mapping from `breast-level_annotations.csv`
- `breast_density` encoded to ordinal (A=1, B=2, C=3, D=4)
- Guaranteed 109-dim tabular vector (107 pyradiomics placeholders + age_norm + density_encoded)
- Robust image loading: `study_id/image_id.png` with mid-gray fallback
- `StandardScaler` fit strictly on train fold
- Safe MC Dropout (`enable_only_dropout`) — BatchNorm stays in `.eval()`
- Differential LR: `1e-5` backbone / `1e-3` heads
- Focal Loss (gamma=2) + dynamic class weights
- StratifiedGroupKFold(n_splits=5) + zero-leakage assertion


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 1 | IMPORTS, SEEDS & DEVICE
# ─────────────────────────────────────────────────────────────────────
import os, sys, random, warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import roc_auc_score, accuracy_score, recall_score

warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'[GPU] {torch.cuda.get_device_name(0)}  |  '
          f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
    print('[MPS] Apple Silicon GPU')
else:
    device = torch.device('cpu')
    print('[CPU] No GPU detected — training will be slow')

print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 2 | CSV INGESTION & LABEL ENGINEERING
# ─────────────────────────────────────────────────────────────────────
IMG_DIR  = Path('./images_png')
TAB_DIM  = 109   # 107 pyradiomics + age_norm + density_encoded

# ── 1. Load CSV ───────────────────────────────────────────────────────────
breast_csv  = Path('breast-level_annotations.csv')
finding_csv = Path('finding_annotations.csv')

if breast_csv.exists():
    df_raw = pd.read_csv(breast_csv)
    print(f'[OK] Loaded {breast_csv.name}: {len(df_raw):,} rows, {df_raw.study_id.nunique():,} studies')
elif finding_csv.exists():
    df_raw = pd.read_csv(finding_csv)
    print(f'[OK] Loaded {finding_csv.name}: {len(df_raw):,} rows')
    if 'breast_birads' not in df_raw.columns and 'finding_birads' in df_raw.columns:
        df_raw = df_raw.rename(columns={'finding_birads': 'breast_birads'})
else:
    raise FileNotFoundError(
        'ERROR: Neither breast-level_annotations.csv nor finding_annotations.csv found.\n'
        'Download from https://physionet.org/content/vindr-mammo/1.0.0/')

print(f'  Columns: {list(df_raw.columns)}')

# ── 2. BI-RADS -> Binary Target Label ─────────────────────────────────────
# BI-RADS 1,2 -> 0 (Normal/Benign)  |  BI-RADS 3,4,5 -> 1 (Suspicious)
MALIGNANT_BIRADS = {'BI-RADS 3', 'BI-RADS 4', 'BI-RADS 5', '3', '4', '5'}
df_raw['target_label'] = df_raw['breast_birads'].apply(
    lambda x: 1 if str(x).strip() in MALIGNANT_BIRADS else 0
).astype(int)

print(f'\n[Label Distribution]')
vc = df_raw['target_label'].value_counts().sort_index()
for lbl, cnt in vc.items():
    print(f'  Class {lbl}: {cnt:,} ({cnt/len(df_raw)*100:.1f}%)')

# ── 3. Clinical Metadata Features ─────────────────────────────────────────
# age_norm: VinDr-Mammo does not provide age in breast-level CSV.
#   -> Derive pseudo-age from study_id hash (reproducible, non-zero).
#   -> If you have a separate demographics file, merge it here.
DENSITY_MAP = {'DENSITY A': 1, 'DENSITY B': 2, 'DENSITY C': 3, 'DENSITY D': 4}
df_raw['density_encoded'] = df_raw['breast_density'].map(DENSITY_MAP).fillna(0).astype(float)
df_raw['density_encoded'] = df_raw['density_encoded'] / 4.0   # normalize to [0, 1]

# Reproducible pseudo-age from study_id hash (replace with real age if available)
study_ids_unique = df_raw['study_id'].unique()
age_map = {sid: (hash(sid) % 40 + 40) / 100.0 for sid in study_ids_unique}  # range 0.40-0.80
df_raw['age_norm'] = df_raw['study_id'].map(age_map)

print(f'\n[Clinical Features]')
print(f'  density_encoded: {df_raw.density_encoded.describe()["min"]:.2f} - {df_raw.density_encoded.describe()["max"]:.2f}')
print(f'  age_norm:        {df_raw.age_norm.min():.2f} - {df_raw.age_norm.max():.2f}')

# ── 4. patient_id assignment ──────────────────────────────────────────────
#  VinDr-Mammo: patient_id = study_id (each study is a unique patient)
for col_try in ['patient_id', 'Patient_ID', 'PatientID']:
    if col_try in df_raw.columns:
        df_raw['patient_id'] = df_raw[col_try]
        break
else:
    df_raw['patient_id'] = df_raw['study_id']   # VinDr standard

# ── 5. Use training split only ────────────────────────────────────────────
if 'split' in df_raw.columns:
    df_train_all = df_raw[df_raw['split'] == 'training'].reset_index(drop=True)
    df_test_held = df_raw[df_raw['split'] == 'test'].reset_index(drop=True)
    print(f'\n[Split] Training: {len(df_train_all):,} | Test (held-out): {len(df_test_held):,}')
else:
    df_train_all = df_raw.reset_index(drop=True)
    df_test_held = pd.DataFrame()
    print(f'\n[Split] No split column found. Using all {len(df_train_all):,} rows for CV.')

print(f'\n[Dataset Ready] {len(df_train_all):,} training images | '
      f'{df_train_all.patient_id.nunique():,} unique patients')


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 3 | TABULAR FEATURE ENGINEERING (rigid 109-dim vector)
# ─────────────────────────────────────────────────────────────────────
# 107 pyradiomics columns + age_norm + density_encoded = 109 features
#
# VinDr-Mammo does NOT ship pre-computed pyradiomics features.
# Options:
#   A) Run PyRadiomics extraction yourself (offline, adds rad_0..rad_106).
#   B) Use view_position & laterality encodings + image stats as surrogates.
#
# This cell implements Option B (reproducible surrogates) so the
# pipeline runs end-to-end immediately. Replace with real radiomics
# by adding columns rad_0..rad_106 to df_train_all and re-running.

df = df_train_all.copy()

# ── Check if real pyradiomics columns exist ───────────────────────────────
real_rad_cols = [f'rad_{i}' for i in range(107) if f'rad_{i}' in df.columns]

if len(real_rad_cols) == 107:
    print(f'[OK] Found 107 real pyradiomics columns.')
else:
    print(f'[INFO] {len(real_rad_cols)}/107 pyradiomics cols found.')
    print(f'       Generating reproducible surrogate features ...')

    # ── Encode categorical columns as surrogate features ──────────────────
    view_map = {'CC': 0.0, 'MLO': 1.0}
    lat_map  = {'L': 0.0, 'R': 1.0}
    df['view_enc'] = df['view_position'].map(view_map).fillna(0.5)
    df['lat_enc']  = df['laterality'].map(lat_map).fillna(0.5)

    # ── Study-level hash -> 107-dim deterministic pseudo-radiomics ────────
    rng = np.random.RandomState(SEED)
    rad_matrix = np.zeros((len(df), 107), dtype=np.float32)
    for idx, sid in enumerate(df['study_id'].values):
        local_rng = np.random.RandomState(abs(hash(sid)) % (2**31))
        rad_matrix[idx] = local_rng.randn(107).astype(np.float32) * 0.5

    for i in range(107):
        df[f'rad_{i}'] = rad_matrix[:, i]
    real_rad_cols = [f'rad_{i}' for i in range(107)]
    print(f'       Generated {len(real_rad_cols)} surrogate radiomics columns.')

# ── Assemble final 109-dim tabular column list ────────────────────────────
tab_cols = real_rad_cols + ['age_norm', 'density_encoded']
assert len(tab_cols) == TAB_DIM, f'Expected {TAB_DIM} features, got {len(tab_cols)}'
print(f'\n[Tabular] Feature vector: {TAB_DIM} dims  '
      f'({len(real_rad_cols)} radiomics + 2 clinical)')
print(f'[Tabular] NaN counts: {df[tab_cols].isna().sum().sum()}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 4 | TRI-MODAL DATASET
# ─────────────────────────────────────────────────────────────────────
class VinDrTriModalDataset(Dataset):
    """
    Loads (image, tabular, label) for each mammogram.
    Image path: images_png/{study_id}/{image_id}.png
               fallback -> images_png/{image_id}.png
               fallback -> mid-gray 512x512 tensor
    """
    def __init__(self, df, img_dir, tab_matrix, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = Path(img_dir)
        self.tab_mat   = np.nan_to_num(
            tab_matrix.astype(np.float32), nan=0.0, posinf=0.0, neginf=0.0)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def _resolve_image(self, row):
        """Return BGR image (H,W,3) or mid-gray fallback."""
        study_id = str(row.get('study_id', ''))
        image_id = str(row['image_id'])
        if not image_id.endswith('.png'):
            image_id += '.png'

        # path 1: images_png/{study_id}/{image_id}.png
        p1 = self.img_dir / study_id / image_id
        # path 2: images_png/{image_id}.png
        p2 = self.img_dir / image_id

        for p in (p1, p2):
            if p.exists():
                img = cv2.imread(str(p))
                if img is not None:
                    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # fallback: neutral mid-gray (avoids dead-zero gradients)
        return np.full((512, 512, 3), 128, dtype=np.uint8)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = self._resolve_image(row)

        if self.transform:
            image = self.transform(image=image)['image']

        tab   = torch.tensor(self.tab_mat[idx], dtype=torch.float32)
        label = torch.tensor(int(row['target_label']), dtype=torch.long)
        return {'image': image, 'tabular': tab, 'label': label}


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 5 | XAI-MEDCROSSNET++ ARCHITECTURE
# ─────────────────────────────────────────────────────────────────────

class MultiHeadCrossAttention(nn.Module):
    """
    Visual queries attend to tabular keys/values.
    img_feat -> Query   |   tab_feat -> Key, Value
    """
    def __init__(self, embed_dim=256, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn      = nn.MultiheadAttention(
            embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm      = nn.LayerNorm(embed_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, img_feat, tab_feat):
        # img_feat, tab_feat: (B, D)
        Q = img_feat.unsqueeze(1)   # (B, 1, D)
        K = tab_feat.unsqueeze(1)   # (B, 1, D)
        V = tab_feat.unsqueeze(1)   # (B, 1, D)
        attn_out, _ = self.attn(Q, K, V)
        attn_out    = attn_out.squeeze(1)  # (B, D)
        return self.norm(img_feat + self.dropout(attn_out))   # residual


class TabularMLP(nn.Module):
    """Deep MLP for 109-dim tabular input -> 256-dim embedding."""
    def __init__(self, in_dim=109, embed_dim=256, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Linear(256, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, x):
        return self.net(x)


class XAIMedCrossNet(nn.Module):
    """
    XAI-MedCrossNet++
    ├─ Vision: ConvNeXt-Tiny pretrained -> 256-dim
    ├─ Tabular: MLP 109->512->256->256 with BN+GELU+Dropout
    ├─ Fusion: Multi-Head Cross-Attention (4 heads, img Query, tab KV)
    └─ Classifier: MC Dropout -> Linear(256, 2)
    """
    def __init__(self, tab_dim=109, embed_dim=256,
                 num_classes=2, dropout_p=0.3):
        super().__init__()

        # ── Vision backbone ──────────────────────────────────────────────
        backbone = tv_models.convnext_tiny(
            weights=tv_models.ConvNeXt_Tiny_Weights.DEFAULT)
        in_feats = backbone.classifier[2].in_features   # 768
        backbone.classifier[2] = nn.Identity()
        self.backbone = backbone

        self.img_proj = nn.Sequential(
            nn.Linear(in_feats, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
        )

        # ── Tabular sub-network ──────────────────────────────────────────
        self.tab_proj = TabularMLP(tab_dim, embed_dim, dropout_p)

        # ── Multi-Head Cross-Attention fusion ────────────────────────────
        self.cross_attn = MultiHeadCrossAttention(
            embed_dim, num_heads=4, dropout=0.1)

        # ── MC Dropout + Classifier ──────────────────────────────────────
        self.mc_dropout  = nn.Dropout(dropout_p)
        self.classifier  = nn.Linear(embed_dim, num_classes)

        self._init_new_layers()

    def _init_new_layers(self):
        """Xavier init for projection and classification heads."""
        for m in [self.img_proj, self.tab_proj, self.cross_attn, self.classifier]:
            for sub in (m.modules() if hasattr(m, 'modules') else [m]):
                if isinstance(sub, nn.Linear):
                    nn.init.xavier_uniform_(sub.weight)
                    if sub.bias is not None:
                        nn.init.zeros_(sub.bias)

    def forward(self, image, tabular):
        img_emb  = self.img_proj(self.backbone(image))   # (B, 256)
        tab_emb  = self.tab_proj(tabular)                # (B, 256)
        fused    = self.cross_attn(img_emb, tab_emb)     # (B, 256)
        return self.classifier(self.mc_dropout(fused))   # (B, 2)


# ── Focal Loss ───────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.01):
        super().__init__()
        self.gamma           = gamma
        self.weight          = weight
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce   = F.cross_entropy(
            logits, targets,
            weight=self.weight,
            label_smoothing=self.label_smoothing,
            reduction='none')
        pt   = torch.exp(-ce)
        return (((1.0 - pt) ** self.gamma) * ce).mean()


print('[OK] Model architecture defined.')
print(f'     XAIMedCrossNet params: '
      f'{sum(p.numel() for p in XAIMedCrossNet().parameters())/1e6:.1f}M')


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 6 | TRAINING HELPERS & SAFE MC DROPOUT EVALUATION
# ─────────────────────────────────────────────────────────────────────

def enable_only_dropout(model: nn.Module) -> None:
    """
    CRITICAL: Sets model globally to .eval() (freezing BatchNorm/LayerNorm
    running statistics), then selectively enables ONLY nn.Dropout modules
    for Monte Carlo stochastic passes.

    NEVER call model.train() during validation — it corrupts BN statistics
    on small batches, causing random 50% guessing.
    """
    model.eval()
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()


def evaluate_mc_dropout(
        model: nn.Module,
        loader: DataLoader,
        device: torch.device,
        n_samples: int = 10
) -> tuple:
    """
    Monte Carlo Dropout evaluation.
    Returns (accuracy, roc_auc, sensitivity, avg_epistemic_uncertainty)
    """
    enable_only_dropout(model)
    all_preds, all_labels, all_vars = [], [], []

    with torch.no_grad():
        for batch in loader:
            imgs   = batch['image'].to(device, non_blocking=True)
            tabs   = batch['tabular'].to(device, non_blocking=True)
            labels = batch['label'].cpu().numpy()

            # n_samples stochastic forward passes
            mc_probs = np.stack([
                F.softmax(model(imgs, tabs), dim=1)[:, 1].cpu().numpy()
                for _ in range(n_samples)
            ])  # (n_samples, B)

            mean_prob = mc_probs.mean(axis=0)
            var_prob  = mc_probs.var(axis=0)   # epistemic uncertainty

            all_preds.extend(mean_prob)
            all_labels.extend(labels)
            all_vars.extend(var_prob)

    preds  = np.array(all_preds)
    labels = np.array(all_labels)
    binary = (preds >= 0.5).astype(int)

    acc  = accuracy_score(labels, binary)
    auc  = roc_auc_score(labels, preds) if len(np.unique(labels)) > 1 else 0.5
    sens = recall_score(labels, binary, zero_division=0)   # sensitivity
    unc  = float(np.mean(all_vars))                        # avg uncertainty
    return acc, auc, sens, unc


def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    """Differential learning rates: backbone 1e-5, heads 1e-3."""
    return torch.optim.AdamW([
        {'params': model.backbone.parameters(),   'lr': 1e-5,  'name': 'backbone'},
        {'params': model.img_proj.parameters(),   'lr': 1e-3,  'name': 'img_proj'},
        {'params': model.tab_proj.parameters(),   'lr': 1e-3,  'name': 'tab_proj'},
        {'params': model.cross_attn.parameters(), 'lr': 1e-3,  'name': 'cross_attn'},
        {'params': model.classifier.parameters(), 'lr': 1e-3,  'name': 'classifier'},
    ], weight_decay=1e-2)


def build_criterion(train_labels: np.ndarray, device: torch.device) -> FocalLoss:
    """Focal Loss with inverse-frequency class weights."""
    counts = np.bincount(train_labels, minlength=2).astype(float)
    total  = counts.sum()
    weight = torch.tensor(
        [total / (2 * max(counts[0], 1)),
         total / (2 * max(counts[1], 1))],
        dtype=torch.float32).to(device)
    return FocalLoss(gamma=2.0, weight=weight, label_smoothing=0.01)


print('[OK] Training helpers defined.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 7 | ALBUMENTATIONS TRANSFORMS
# ─────────────────────────────────────────────────────────────────────

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.1),
    A.RandomBrightnessContrast(brightness_limit=0.15,
                               contrast_limit=0.15, p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05,
                       rotate_limit=10, p=0.3),
    A.GaussNoise(var_limit=(5.0, 30.0), p=0.2),
    A.CLAHE(clip_limit=2.0, p=0.2),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

print('[OK] Transforms ready.')
print(f'     Train: {len(train_transform.transforms)} augmentation steps')
print(f'     Val:   {len(val_transform.transforms)} steps (resize + normalize only)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 8 | FULL PIPELINE — StratifiedGroupKFold x 5
# ─────────────────────────────────────────────────────────────────────

# ── Hyperparameters ───────────────────────────────────────────────────────
EPOCHS      = 20
BATCH_SIZE  = 8
NUM_WORKERS = 0        # keep 0 on Windows
N_FOLDS     = 5
MC_SAMPLES  = 10
EMBED_DIM   = 256
DROPOUT_P   = 0.3

sgkf      = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_aucs = []
fold_best = []

for fold, (tr_idx, vl_idx) in enumerate(
        sgkf.split(df, df['target_label'], df['patient_id'])):

    print(f'\n{"="*65}')
    print(f'  FOLD {fold+1}/{N_FOLDS}   |   '
          f'{len(tr_idx):,} train images  /  {len(vl_idx):,} val images')
    print(f'  Train positives: '
          f'{df.iloc[tr_idx]["target_label"].sum():,} '
          f'({df.iloc[tr_idx]["target_label"].mean()*100:.1f}%)  |  '
          f'Val positives: '
          f'{df.iloc[vl_idx]["target_label"].sum():,} '
          f'({df.iloc[vl_idx]["target_label"].mean()*100:.1f}%)')
    print(f'  Train patients: {df.iloc[tr_idx]["patient_id"].nunique():,} '
          f'| Val patients: {df.iloc[vl_idx]["patient_id"].nunique():,}')
    print(f'{"="*65}')

    tr_df = df.iloc[tr_idx]
    vl_df = df.iloc[vl_idx]

    # ── ZERO-LEAKAGE ASSERTION ───────────────────────────────────────────
    train_pats = set(tr_df['patient_id'])
    val_pats   = set(vl_df['patient_id'])
    assert len(train_pats & val_pats) == 0, \
        f'[FATAL] Patient leakage detected: {train_pats & val_pats}'
    print('[OK] Zero patient overlap verified.')

    # ── STANDARD SCALER — fit on train ONLY ─────────────────────────────
    scaler = StandardScaler()
    tr_tab_raw = np.nan_to_num(tr_df[tab_cols].fillna(0).values, nan=0.0)
    vl_tab_raw = np.nan_to_num(vl_df[tab_cols].fillna(0).values, nan=0.0)
    tr_tab_scaled = scaler.fit_transform(tr_tab_raw)
    vl_tab_scaled = scaler.transform(vl_tab_raw)

    # ── DATA LOADERS ─────────────────────────────────────────────────────
    tr_ds = VinDrTriModalDataset(tr_df, IMG_DIR, tr_tab_scaled, train_transform)
    vl_ds = VinDrTriModalDataset(vl_df, IMG_DIR, vl_tab_scaled, val_transform)

    tr_loader = DataLoader(
        tr_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'),
        drop_last=True)
    vl_loader = DataLoader(
        vl_ds, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=(device.type == 'cuda'))

    # ── MODEL ─────────────────────────────────────────────────────────────
    model     = XAIMedCrossNet(tab_dim=TAB_DIM,
                               embed_dim=EMBED_DIM,
                               dropout_p=DROPOUT_P).to(device)
    optimizer = build_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=EPOCHS, eta_min=1e-7)
    criterion = build_criterion(tr_df['target_label'].values, device)

    print(f'[Training] Starting {EPOCHS}-epoch run ...')
    print(f'  {"Ep":>3}  {"Loss":>7}  {"TrAcc":>6}  '
          f'{"VlAcc":>6}  {"AUC":>6}  {"Sens":>6}  {"Unc":>8}  Note')
    print(f'  {"-"*75}')

    best_auc, best_epoch = 0.0, 0

    for epoch in range(1, EPOCHS + 1):

        # ── TRAIN ────────────────────────────────────────────────────────
        model.train()
        running_loss = 0.0
        correct      = 0
        total        = 0

        for batch in tr_loader:
            imgs   = batch['image'].to(device, non_blocking=True)
            tabs   = batch['tabular'].to(device, non_blocking=True)
            labels = batch['label'].to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(imgs, tabs)
            loss   = criterion(logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * labels.size(0)
            correct      += (logits.argmax(1) == labels).sum().item()
            total        += labels.size(0)

        scheduler.step()
        epoch_loss = running_loss / total
        epoch_acc  = correct / total

        # ── VALIDATE (safe MC Dropout) ────────────────────────────────────
        vl_acc, vl_auc, sens, unc = evaluate_mc_dropout(
            model, vl_loader, device, n_samples=MC_SAMPLES)

        # ── CHECKPOINT ───────────────────────────────────────────────────
        note = ''
        if vl_auc > best_auc:
            best_auc, best_epoch = vl_auc, epoch
            torch.save({
                'epoch':      epoch,
                'fold':       fold + 1,
                'state_dict': model.state_dict(),
                'optimizer':  optimizer.state_dict(),
                'val_auc':    vl_auc,
                'val_acc':    vl_acc,
                'scaler':     scaler,
                'tab_cols':   tab_cols,
            }, 'best_vindr_model.pth')
            note = 'SAVED'

        print(f'  {epoch:3d}  {epoch_loss:7.4f}  '
              f'{epoch_acc*100:6.2f}%  '
              f'{vl_acc*100:6.2f}%  '
              f'{vl_auc:6.4f}  '
              f'{sens:6.4f}  '
              f'{unc:8.6f}  '
              f'{note}')

    print(f'\n  Fold {fold+1} | Best Val AUC: {best_auc:.4f}  (epoch {best_epoch})')
    fold_aucs.append(best_auc)
    fold_best.append({'fold': fold+1, 'auc': best_auc, 'epoch': best_epoch})

    break   # <-- REMOVE this line to run all 5 folds

print(f'\n{"="*65}')
print(f'  Cross-Validation Summary')
for r in fold_best:
    print(f'    Fold {r["fold"]}:  AUC = {r["auc"]:.4f}  (best epoch {r["epoch"]})')
print(f'  Mean AUC: {np.mean(fold_aucs):.4f} +/- {np.std(fold_aucs):.4f}')
print(f'  Best checkpoint: best_vindr_model.pth')
print(f'{"="*65}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────
# CELL 9 | INFERENCE ON HELD-OUT TEST SET
# ─────────────────────────────────────────────────────────────────────

if len(df_test_held) == 0:
    print('No held-out test set found. Skipping inference.')
else:
    print(f'[Test Set] {len(df_test_held):,} images | '
          f'{df_test_held.patient_id.nunique():,} patients')

    # Load best checkpoint
    ckpt    = torch.load('best_vindr_model.pth', map_location=device)
    model   = XAIMedCrossNet(tab_dim=TAB_DIM,
                              embed_dim=EMBED_DIM,
                              dropout_p=DROPOUT_P).to(device)
    model.load_state_dict(ckpt['state_dict'])
    scaler_test = ckpt['scaler']
    tab_cols_ck = ckpt['tab_cols']
    print(f'[OK] Loaded checkpoint from fold {ckpt["fold"]}, '
          f'epoch {ckpt["epoch"]}, Val AUC {ckpt["val_auc"]:.4f}')

    # Build test tabular features (must match training columns)
    #  -> apply same surrogate generation if needed
    df_test = df_test_held.copy()
    for col in tab_cols_ck:
        if col not in df_test.columns:
            if col.startswith('rad_'):
                i = int(col.split('_')[1])
                df_test[col] = df_test['study_id'].apply(
                    lambda s, idx=i: np.random.RandomState(
                        abs(hash(s)) % (2**31)).randn(107)[idx] * 0.5)
            else:
                df_test[col] = 0.0

    test_tab_raw    = np.nan_to_num(df_test[tab_cols_ck].fillna(0).values)
    test_tab_scaled = scaler_test.transform(test_tab_raw)

    test_ds     = VinDrTriModalDataset(df_test, IMG_DIR,
                                        test_tab_scaled, val_transform)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=0)

    test_acc, test_auc, test_sens, test_unc = evaluate_mc_dropout(
        model, test_loader, device, n_samples=MC_SAMPLES)

    print(f'\n[Test Results]')
    print(f'  Accuracy    : {test_acc*100:.2f}%')
    print(f'  ROC-AUC     : {test_auc:.4f}')
    print(f'  Sensitivity : {test_sens:.4f}')
    print(f'  Uncertainty : {test_unc:.6f}')
